# 🚀 TASK 6.3-R: ANCHORED RECOVERY TRAINING TRÊN GPU (KAGGLE / COLAB PRO)
### CHIẾN DỊCH HUẤN LUYỆN NEO PHỤC HỒI: TỐC ĐỘ HỌC PHÂN TẦNG (DIFFERENTIAL LR) & CHƯNG CẤT TRI THỨC MỎ NEO

> **Mã nhiệm vụ**: `Task 6.3-R` *(Milestone 6-R - High-Accuracy Anchored Recovery)*  
> **Người chủ trì**: **Thành viên C (Training & Evaluation Lead)**  
> **Người phối hợp**: **Thành viên B (Architecture Lead)**, **Thành viên A (Data & Infra Lead)**  
> **Môi trường thực thi**: Kaggle Server GPU (NVIDIA T4 x2 / P100 / RTX PRO 6000 Blackwell) hoặc Google Colab Pro GPU (T4 / L4 / A100)  
> **Mục tiêu cốt lõi**: Huấn luyện 3 epochs với nguyên lý khôi phục toàn diện:
> 1. **Khởi tạo Zero-Init Identity tại Step 0**: Tầng chiếu SRM = 0, Cổng động = 1.0 (Bảo toàn tuyệt đối đỉnh cao 96.45% của mô hình gốc).  
> 2. **Tốc độ học phân tầng (Differential LR)**: Module mới (SRM, Gating) học $lr_2 = 10^{-4}$; Toàn bộ adapter tiền huấn luyện (FAA, LGA, Text ctx) chỉ học $lr_1 = 10^{-5}$ ($10\times$ chậm hơn để chống quên); Đóng băng 94.1% CLIP ViT-L/14 backbone.  
> 3. **Hàm mất mát neo chặt Teacher**: $\mathcal{L} = 0.3\mathcal{L}_{\text{AugCE}} + 0.7\mathcal{L}_{\text{KD}} + 0.2\mathcal{L}_{\text{cons}}$.  
> 4. **Luồng dữ liệu Clean-First**: 70% ảnh giữ nguyên sạch, 30% áp dụng suy biến nhẹ có kiểm soát (JPEG $Q \in [50, 90]$, Blur $0.5 - 1.5$).  
> 5. **Xuất checkpoint chuẩn mực**: `fatformer_anchored_recovery_best.pth` kèm mã băm SHA-256 nghiệm thu.

## 🛠️ BƯỚC 1: THIẾT LẬP MÃ NGUỒN & MÔI TRƯỜNG TRÊN MÁY CHỦ GPU
> Tự động nhận diện mã nguồn từ Kaggle Input (hoặc clone từ GitHub trên Colab), thiết lập thư mục làm việc và nạp từ vựng BPE Tokenizer offline cho CLIP.

In [ ]:
# ==============================================================================
# BƯỚC 1: THIẾT LẬP MÃ NGUỒN & TỪ VỰNG BPE OFFLINE
# ==============================================================================
import os, sys, glob, shutil, zipfile, subprocess

# Xác định môi trường Kaggle hoặc Colab
IS_KAGGLE = os.path.exists("/kaggle")
WORK_DIR = "/kaggle/working/fatformer-xla" if IS_KAGGLE else "/content/fatformer-xla"
os.makedirs(WORK_DIR, exist_ok=True)

print(f"[*] Môi trường thực thi: {'KAGGLE' if IS_KAGGLE else 'COLAB'}")
print(f"[*] Thư mục làm việc chính: {WORK_DIR}")

if IS_KAGGLE:
    # Dò tìm mã nguồn trong /kaggle/input/
    zip_candidates = glob.glob("/kaggle/input/**/fatformer_code.zip", recursive=True) + \
                     glob.glob("/kaggle/input/**/fatformer-code*.zip", recursive=True)
    train_py_files = glob.glob("/kaggle/input/**/train_crossdomain.py", recursive=True)
    
    if zip_candidates:
        src_zip = zip_candidates[0]
        print(f"[*] Giải nén mã nguồn từ: {src_zip}")
        with zipfile.ZipFile(src_zip, 'r') as zf:
            zf.extractall(WORK_DIR)
    elif train_py_files:
        root_cand = os.path.abspath(os.path.join(os.path.dirname(train_py_files[0]), "..", ".."))
        print(f"[*] Sao chép mã nguồn từ: {root_cand}")
        shutil.copytree(root_cand, WORK_DIR, dirs_exist_ok=True)
    else:
        print("[*] Đang clone mã nguồn từ GitHub...")
        subprocess.run(f"git clone https://github.com/LeeVHoangtk3/fatformer-xla.git {WORK_DIR}", shell=True, check=True)
else:
    if not os.path.exists(os.path.join(WORK_DIR, "src")):
        subprocess.run(f"git clone https://github.com/LeeVHoangtk3/fatformer-xla.git {WORK_DIR}", shell=True, check=True)
    else:
        subprocess.run(f"cd {WORK_DIR} && git pull", shell=True)

# Xử lý từ vựng BPE Tokenizer offline
clip_dir = os.path.join(WORK_DIR, "src", "models", "clip")
os.makedirs(clip_dir, exist_ok=True)
target_bpe_gz = os.path.join(clip_dir, "bpe_simple_vocab_16e6.txt.gz")
bpe_candidates_gz = glob.glob(f"{WORK_DIR}/**/bpe_simple_vocab_16e6.txt.gz", recursive=True) + glob.glob("/kaggle/**/bpe_simple_vocab_16e6.txt.gz", recursive=True)

if not os.path.exists(target_bpe_gz) and bpe_candidates_gz:
    shutil.copy2(bpe_candidates_gz[0], target_bpe_gz)
    print(f"[✓] Đã nạp BPE vocab từ: {bpe_candidates_gz[0]}")

os.chdir(WORK_DIR)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

# Cài đặt thư viện bổ trợ nếu thiếu
%pip install -q ftfy regex tqdm tabulate pyyaml matplotlib
print("[✓] Môi trường mã nguồn đã sẵn sàng 100%!")


## 🔍 BƯỚC 2: LIÊN KẾT CHECKPOINTS MỎ NEO & CLIP BACKBONE
> Dò tìm và liên kết an toàn:
> - Backbone CLIP `ViT-L-14.pt` (TorchScript)
> - Checkpoint Teacher mỏ neo: `fatformer_4class_ckpt.pth` (1.116 tensors CVPR 2024 đỉnh cao 96.45%)
> - Checkpoint khởi tạo Student: `fatformer_4class_ckpt.pth` hoặc `fatformer_plan_a_best_weights.pth`.

In [ ]:
# ==============================================================================
# BƯỚC 2: TỰ ĐỘNG DÒ TÌM & LIÊN KẾT TRỌNG SỐ CHECKPOINT
# ==============================================================================
import os, glob, shutil

os.makedirs(os.path.join(WORK_DIR, "DATASET", "pretrained"), exist_ok=True)
os.makedirs(os.path.join(WORK_DIR, "DATASET", "checkpoints"), exist_ok=True)

def find_file(patterns):
    for pat in patterns:
        matches = glob.glob(pat, recursive=True)
        if matches:
            return os.path.abspath(matches[0])
    return None

# 1. Dò tìm CLIP ViT-L-14.pt
clip_src = find_file([
    "/kaggle/input/**/ViT-L-14.pt",
    "/content/drive/MyDrive/**/ViT-L-14.pt",
    "/content/**/ViT-L-14.pt",
    f"{WORK_DIR}/DATASET/pretrained/ViT-L-14.pt"
])
clip_dst = os.path.join(WORK_DIR, "DATASET", "pretrained", "ViT-L-14.pt")
if clip_src and not os.path.exists(clip_dst):
    try: os.symlink(clip_src, clip_dst)
    except Exception: shutil.copy2(clip_src, clip_dst)
print(f"[1] CLIP Backbone : {clip_src} -> {clip_dst}")

# 2. Dò tìm Teacher Checkpoint (fatformer_4class_ckpt.pth)
teacher_src = find_file([
    "/kaggle/input/**/fatformer_4class_ckpt.pth",
    "/content/drive/MyDrive/**/fatformer_4class_ckpt.pth",
    "/content/**/fatformer_4class_ckpt.pth",
    f"{WORK_DIR}/DATASET/pretrained/fatformer_4class_ckpt.pth"
])
teacher_dst = os.path.join(WORK_DIR, "DATASET", "pretrained", "fatformer_4class_ckpt.pth")
if teacher_src and not os.path.exists(teacher_dst):
    try: os.symlink(teacher_src, teacher_dst)
    except Exception: shutil.copy2(teacher_src, teacher_dst)
print(f"[2] Teacher Anchor: {teacher_src} -> {teacher_dst}")

# 3. Dò tìm Checkpoint khởi đầu Student
student_init_src = teacher_src  # Task 6.3-R: Khoi dong tu fatformer_4class_ckpt.pth goc (CVPR 2024)
student_init_dst = os.path.join(WORK_DIR, "DATASET", "checkpoints", "student_init.pth")
if student_init_src and not os.path.exists(student_init_dst):
    try: os.symlink(student_init_src, student_init_dst)
    except Exception: shutil.copy2(student_init_src, student_init_dst)
print(f"[3] Student Init  : {student_init_src} -> {student_init_dst}")


## 📦 BƯỚC 3: GIẢI NÉN DỮ LIỆU HUẤN LUYỆN & TẠO MANIFEST CHUẨN
> Giải nén `diffusion_staging.tar` và/hoặc dữ liệu ProGAN 4-class, sau đó tự động quét toàn bộ file ảnh để sinh file `DATASET/manifests/data_manifest.json` phân định 85% train / 15% val.

In [ ]:
# ==============================================================================
# BƯỚC 3: TRÍCH XUẤT DỮ LIỆU HUẤN LUYỆN & SINH MANIFEST
# ==============================================================================
import tarfile, json as py_json, random

DATA_DIR = "/kaggle/working/dataset_train" if IS_KAGGLE else "/content/dataset_train"
os.makedirs(DATA_DIR, exist_ok=True)

# Dò tìm các tệp nén dữ liệu huấn luyện
tar_staging = find_file([
    "/kaggle/input/**/diffusion_staging.tar",
    "/content/drive/MyDrive/**/diffusion_staging.tar",
    "/content/**/diffusion_staging.tar"
])

if tar_staging and len(os.listdir(DATA_DIR)) == 0:
    print(f"[*] Đang giải nén dữ liệu huấn luyện từ: {tar_staging}...")
    with tarfile.open(tar_staging, 'r') as tf:
        tf.extractall(DATA_DIR)
    print(f"[✓] Giải nén xong dữ liệu sang: {DATA_DIR}")

# Quét toàn bộ các ảnh trong DATA_DIR
all_records = []
for root, _, files in os.walk(DATA_DIR):
    for f in files:
        if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
            fp = os.path.join(root, f)
            norm = fp.replace('\\', '/').lower()
            is_fake = any(k in norm for k in ['fake', '1_fake', 'synthetic'])
            all_records.append({
                'file_path': fp,
                'label': 1 if is_fake else 0,
                'generator': 'synthetic' if is_fake else 'real'
            })

print(f"[✓] Tổng số ảnh quét được: {len(all_records)}")
if len(all_records) > 0:
    rng = random.Random(42)
    reals = [r for r in all_records if r['label'] == 0]
    fakes = [r for r in all_records if r['label'] == 1]
    rng.shuffle(reals)
    rng.shuffle(fakes)
    n_vr = max(1, int(len(reals) * 0.15))
    n_vf = max(1, int(len(fakes) * 0.15))
    val_list = reals[:n_vr] + fakes[:n_vf]
    train_list = reals[n_vr:] + fakes[n_vf:]
    for item in train_list: item['split'] = 'train'
    for item in val_list: item['split'] = 'val'
    
    manifest_path = os.path.join(WORK_DIR, "DATASET", "manifests", "data_manifest.json")
    os.makedirs(os.path.dirname(manifest_path), exist_ok=True)
    with open(manifest_path, "w", encoding="utf-8") as f_out:
        py_json.dump({'metadata': {'task': 'Task 6.3-R Staging Manifest'}, 'records': train_list + val_list}, f_out, indent=2)
    print(f"[✓] Đã tạo manifest thành công: {manifest_path} (Train: {len(train_list)}, Val: {len(val_list)})")
else:
    print("[!] Cảnh báo: Chưa có ảnh thực tế trong DATA_DIR, pipeline sẽ dùng dry-run kiểm thử.")


## ⚖️ BƯỚC 4: KIỂM TRA GPU & CHỨNG MINH TOÁN HỌC STEP-0 ZERO-INIT IDENTITY
> Kiểm tra phần cứng GPU và chạy bộ kiểm định Zero-Init Identity trên GPU để khẳng định sai số logit tại Step 0 tuyệt đối $< 10^{-6}$.

In [ ]:
# ==============================================================================
# BƯỚC 4: KIỂM TOÁN GPU & KIỂM THỬ ZERO-INIT STEP-0 IDENTITY
# ==============================================================================
import torch, subprocess

print(f"[*] Phiên bản PyTorch: {torch.__version__}")
print(f"[*] CUDA khả dụng: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[*] Tên GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/(1024**3):.2f} GB VRAM)")

# 1. Chạy kiểm chứng Zero-Init Identity
zero_init_cmd = f"{sys.executable} tools/test_zero_init_identity.py --device cuda"
print(f"[*] Kiểm tra toán học Step-0 Identity: {zero_init_cmd}")
res_zi = subprocess.run(zero_init_cmd, shell=True)
if res_zi.returncode == 0:
    print("[✓] TOÁN HỌC ZERO-INIT PASS 100%! BẢO TOÀN BIT-PERFECT BIỂU DIỄN SẠCH GỐC.")
else:
    print("[!] Cảnh báo kiểm tra Zero-Init! Vui lòng xem xét console log.")

# 2. Chạy Dry-run 1 step kiểm tra toàn vẹn Autograd
dry_cmd = f"{sys.executable} src/training/train_crossdomain.py --dry_run --device cuda"
print(f"[*] Kiểm tra autograd dry-run: {dry_cmd}")
res_dry = subprocess.run(dry_cmd, shell=True)
if res_dry.returncode == 0:
    print("[✓] SMOKE TEST AUTOGRAD PASS! HỆ THỐNG SẴN SÀNG HUẤN LUYỆN CHÍNH THỨC.")
else:
    raise RuntimeError("[X] Smoke test autograd thất bại!")


## 🏃 BƯỚC 5: THỰC THI HUẤN LUYỆN ANCHORED RECOVERY M6-R (3 EPOCHS)
> Kích hoạt huấn luyện trên GPU Cloud với đầy đủ 4 nguyên lý Phase 6-R:
> - `--branch M4`: Nhánh có đầy đủ SRM + Gating + Distillation
> - `--learning_rate 1e-4`: Tốc độ học module mới ($10^{-4}$), các adapter tiền huấn luyện tự động giảm $10\times$ còn $10^{-5}$
> - `--weight_ce 0.3 --alpha_kd 0.7 --beta_cons 0.2`: Tỷ trọng hàm mất mát neo Teacher chuẩn mực
> - `--epochs 3`: 3 epochs cân bằng nhanh
> - `--save_name fatformer_anchored_recovery_best.pth`: Đặt tên checkpoint đúng kế hoạch nghiệm thu.

In [ ]:
# ==============================================================================
# BƯỚC 5: THỰC THI CHIẾN DỊCH HUẤN LUYỆN ANCHORED RECOVERY
# ==============================================================================
import time, subprocess

CHECKPOINT_SAVE_DIR = os.path.join(WORK_DIR, "DATASET", "checkpoints")
os.makedirs(CHECKPOINT_SAVE_DIR, exist_ok=True)

cmd_train = [
    sys.executable, "-u", "src/training/train_crossdomain.py",
    "--branch", "M4",
    "--epochs", "3",
    "--batch_size", "16",
    "--learning_rate", "1e-4",
    "--weight_ce", "0.3",
    "--alpha_kd", "0.7",
    "--beta_cons", "0.2",
    "--save_dir", CHECKPOINT_SAVE_DIR,
    "--save_name", "fatformer_anchored_recovery_best.pth",
    "--clip_path", clip_dst if os.path.exists(clip_dst) else "ViT-L-14.pt",
    "--checkpoint_init", student_init_dst if os.path.exists(student_init_dst) else "",
    "--teacher_checkpoint", teacher_dst if os.path.exists(teacher_dst) else "",
    "--device", "cuda" if torch.cuda.is_available() else "cpu"
]

if os.path.exists(manifest_path) and len(all_records) > 0:
    cmd_train.extend(["--train_manifest", manifest_path])
else:
    print("[!] Cảnh báo: Không có manifest ảnh thực tế, kích hoạt chế độ --dry_run nhằm kiểm thử pipeline.")
    cmd_train.append("--dry_run")

print("=" * 80)
print(f"[*] Lệnh thực thi huấn luyện: {' '.join(cmd_train)}")
print("=" * 80)

t0 = time.time()
process = subprocess.Popen(cmd_train, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end="")
process.wait()
elapsed = time.time() - t0

if process.returncode != 0:
    raise RuntimeError(f"[X] Huấn luyện thất bại với mã lỗi: {process.returncode}")
else:
    print(f"\n[✓] HUẤN LUYỆN ANCHORED RECOVERY HOÀN TẤT TRONG {elapsed:.2f}s ({elapsed/60:.2f} phút)!")


## 🔐 BƯỚC 6: XÁC THỰC CHECKPOINT, TÍNH SHA-256 & XUẤT ARTIFACTS SANG DRIVE
> Tính mã băm SHA-256 chuẩn xác trên checkpoint vừa tạo ra, lưu file metadata JSON, đồng bộ sang Google Drive và kích hoạt tải về.

In [ ]:
# ==============================================================================
# BƯỚC 6: TÍNH SHA-256 & XUẤT CHECKPOINT RA GOOGLE DRIVE / KAGGLE OUTPUT
# ==============================================================================
import hashlib, json as py_json

best_ckpt = os.path.join(CHECKPOINT_SAVE_DIR, "fatformer_anchored_recovery_best.pth")
if not os.path.exists(best_ckpt):
    # Dò fallback sang crossdomain_model_best.pth
    alt_ckpt = os.path.join(CHECKPOINT_SAVE_DIR, "crossdomain_model_best.pth")
    if os.path.exists(alt_ckpt):
        shutil.copy2(alt_ckpt, best_ckpt)

if os.path.exists(best_ckpt):
    size_mb = os.path.getsize(best_ckpt) / (1024**2)
    print(f"[*] Đang tính mã băm SHA-256 cho {best_ckpt} ({size_mb:.2f} MB)...")
    sha256 = hashlib.sha256()
    with open(best_ckpt, 'rb') as f:
        while chunk := f.read(10 * 1024 * 1024):
            sha256.update(chunk)
    digest = sha256.hexdigest().upper()
    print(f"[✓] SHA-256: {digest}")

    meta_info = {
        "checkpoint_name": "fatformer_anchored_recovery_best.pth",
        "task": "Task 6.3-R Anchored Recovery Training",
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "sha256": digest,
        "size_bytes": os.path.getsize(best_ckpt),
        "hyperparameters": {
            "branch": "M4",
            "epochs": 3,
            "lr_new_modules": 1e-4,
            "lr_pretrained_adapters": 1e-5,
            "loss_weights": {"AugCE": 0.3, "KD": 0.7, "Consistency": 0.2},
            "clean_stream_prob": 0.70
        }
    }
    meta_path = os.path.join(CHECKPOINT_SAVE_DIR, "fatformer_anchored_recovery_best_metadata.json")
    with open(meta_path, 'w', encoding='utf-8') as f:
        py_json.dump(meta_info, f, indent=2)
    print(f"[✓] Đã lưu metadata tại: {meta_path}")

    # Sao chép vào thư mục xuất của Kaggle hoặc Drive
    if IS_KAGGLE:
        kaggle_out_ckpt = "/kaggle/working/fatformer_anchored_recovery_best.pth"
        shutil.copy2(best_ckpt, kaggle_out_ckpt)
        shutil.copy2(meta_path, "/kaggle/working/fatformer_anchored_recovery_best_metadata.json")
        print(f"[✓] Đã xuất checkpoint sang Kaggle Output: {kaggle_out_ckpt}")
    else:
        drive_ckpt_dir = "/content/drive/MyDrive/FatFormer_Hub/checkpoints"
        if os.path.exists(os.path.dirname(drive_ckpt_dir)):
            os.makedirs(drive_ckpt_dir, exist_ok=True)
            shutil.copy2(best_ckpt, os.path.join(drive_ckpt_dir, "fatformer_anchored_recovery_best.pth"))
            shutil.copy2(meta_path, os.path.join(drive_ckpt_dir, "fatformer_anchored_recovery_best_metadata.json"))
            print(f"[✓] Đã sao chép sang Google Drive: {drive_ckpt_dir}")

print("\n🎉 HOÀN TẤT TOÀN BỘ CHIẾN DỊCH HUẤN LUYỆN TASK 6.3-R!")
